# WNAM CleanDetrend NEU → xarray

Notebook to load WNAM CleanDetrend NEU files, subset by catalog, and build an xarray.Dataset (mirrors the UNR workflow).

## Imports

In [32]:

from __future__ import annotations

import re
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import xarray as xr
from io import StringIO


## Config

In [23]:
DATA_DIR = Path("./data/WNAM_Clean_DetrendNeuTimeSeries_comb_20260107")
CATALOG_CSV = Path("./resources/catalog_subset_bbox.csv")
OUT_NC = Path("./data/wnam_clean_detrend_NEU_subset_bbox.nc")


## Load allowed stations from catalog

In [24]:

def load_allowed_stations(csv_path: Path) -> List[str]:
    df = pd.read_csv(csv_path)
    candidates = ["station", "site", "name", "sta", "stid", "id"]
    cols_lower = {c.lower(): c for c in df.columns}
    chosen = None
    for key in candidates:
        if key in cols_lower:
            chosen = cols_lower[key]
            break
    if chosen is None:
        obj_cols = [c for c in df.columns if df[c].dtype == object]
        if not obj_cols:
            raise ValueError("No station column found")
        chosen = obj_cols[0]
    return (
        df[chosen].astype(str).str.strip().str.lower()
        .replace({"nan": np.nan}).dropna().unique().tolist()
    )

allowed_stations = load_allowed_stations(CATALOG_CSV)
print(f"Allowed stations: {len(allowed_stations)}")


Allowed stations: 604


## Helpers

In [25]:

def station_from_filename(p: Path) -> Optional[str]:
    m = re.match(r"^([A-Za-z0-9]+)CleanDetrend\.(?:enu|neu|NEU|ENU)$", p.name)
    if m:
        return m.group(1).lower()
    if "CleanDetrend" in p.name:
        return p.name.split("CleanDetrend", 1)[0].lower()
    return None

def decimal_year_to_datetime(dy: np.ndarray) -> pd.DatetimeIndex:
    dy = np.asarray(dy, dtype=float)
    year = np.floor(dy).astype(int)
    frac = dy - year
    start = pd.to_datetime(year.astype(str) + "-01-01")
    end = pd.to_datetime((year + 1).astype(str) + "-01-01")
    return pd.DatetimeIndex(start + (end - start) * frac)


## Reader

In [36]:
@dataclass
class StationMeta:
    station: str
    latitude_dd: Optional[float] = None
    east_longitude_dd: Optional[float] = None
    height_m: Optional[float] = None
    ref_x_m: Optional[float] = None
    ref_y_m: Optional[float] = None
    ref_z_m: Optional[float] = None
    file_path: Optional[str] = None

HEADER_FLOAT_PATTERNS = {
    "latitude_dd": re.compile(r"^#\s*Latitude\(DD\)\s*:\s*([-\d\.]+)"),
    "east_longitude_dd": re.compile(r"^#\s*East Longitude\(DD\)\s*:\s*([-\d\.]+)"),
    "height_m": re.compile(r"^#\s*Height\s*\(M\)\s*:\s*([-\d\.]+)"),
    "ref_x_m": re.compile(r"^#\s*Reference_X\s*:\s*([-\d\.]+)"),
    "ref_y_m": re.compile(r"^#\s*Reference_Y\s*:\s*([-\d\.]+)"),
    "ref_z_m": re.compile(r"^#\s*Reference_Z\s*:\s*([-\d\.]+)"),
}

def parse_gps_file(filepath):
    """
    Parse GPS time series file and return data as DataFrame.

    Parameters:
    -----------
    filepath : str
        Path to the GPS time series file

    Returns:
    --------
    pandas DataFrame with time series data
    """

    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Find the start of data section
    data_start = None
    for i, line in enumerate(lines):
        if line.strip().startswith('# Dec Yr'):
            data_start = i + 1
            break

    # Parse data section
    if data_start:
        data_lines = [line.strip() for line in lines[data_start:]
                     if line.strip() and not line.startswith('#')]

        columns = ['Dec_Yr', 'Yr', 'DayOfYr', 'N', 'E', 'U',
                   'N_sig', 'E_sig', 'U_sig', 'CorrNE', 'CorrNU', 'CorrEU', 'Chi_Squared']

        data = []
        for line in data_lines:
            values = line.split()
            if len(values) == len(columns):
                data.append([float(v) for v in values])

        df = pd.DataFrame(data, columns=columns)
    else:
        df = pd.DataFrame()

    return df

In [37]:
parse_gps_file("./data/WNAM_Clean_DetrendNeuTimeSeries_comb_20260107/albhCleanDetrend.neu")

,Dec_Yr,Yr,DayOfYr,N,E,U,N_sig,E_sig,U_sig,CorrNE,CorrNU,CorrEU,Chi_Squared
0,1992.6216,1992.0,228.0,13.92,-2.13,-12.97,4.60,7.41,15.60,0.0059,0.0677,0.0848,3.13
1,1992.6270,1992.0,230.0,-2.03,-15.29,-4.37,3.19,4.37,11.09,-0.0166,-0.0512,0.0884,112.73
2,1992.6298,1992.0,231.0,-13.31,-12.27,4.93,2.68,3.71,9.54,-0.0948,-0.0678,0.2200,101.33
3,1992.6325,1992.0,232.0,-2.39,3.25,-2.47,2.81,3.71,9.02,-0.0645,-0.0769,0.1585,153.74
4,1992.6352,1992.0,233.0,-8.57,-15.63,9.32,2.68,3.57,9.28,-0.0210,-0.0545,0.0890,74.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12057,2025.9767,2025.0,357.0,-1.42,-0.69,-3.39,1.53,2.12,4.25,-0.0260,-0.1590,0.1038,6.64
12058,2025.9795,2025.0,358.0,-2.00,0.33,2.21,1.66,2.38,4.51,0.0390,-0.1431,0.0136,5.40
12059,2025.9822,2025.0,359.0,-0.68,-0.95,-1.60,1.66,2.25,4.38,-0.0026,-0.1840,0.0633,4.00
12060,2025.9849,2025.0,360.0,-1.06,-2.53,0.50,1.53,2.38,4.64,0.0225,-0.1334,0.0121,3.24


## Load, concatenate, save

In [38]:
# Build an index: station -> list of files on disk
all_files = sorted(DATA_DIR.glob("*CleanDetrend.*"))
scanned_files = len(all_files)

files_by_station = {}
unparsed_files = []

for p in all_files:
    st = station_from_filename(p)
    if st is None:
        unparsed_files.append(p.name)
        continue
    files_by_station.setdefault(st, []).append(p)

stations_on_disk = set(files_by_station.keys())

print(f"Files scanned on disk: {scanned_files}")
print(f"Stations found on disk (any): {len(stations_on_disk)}")
print(f"Files with unparseable names: {len(unparsed_files)}")

Files scanned on disk: 2070
Stations found on disk (any): 2070
Files with unparseable names: 0


In [39]:
allowed_set = set(allowed_stations)

missing_on_disk = sorted(allowed_set - stations_on_disk)
extra_on_disk = sorted(stations_on_disk - allowed_set)

print("--------------- Disk vs catalog check ---------------")
print(f"Stations in catalog: {len(allowed_set)}")
print(f"Stations found on disk: {len(stations_on_disk)}")
print(f"Catalog stations missing on disk: {len(missing_on_disk)}")
print(f"Stations on disk but not in catalog: {len(extra_on_disk)}")

if missing_on_disk:
    print("\n--- In catalog but NOT found on disk (first 50) ---")
    for st in missing_on_disk[:50]:
        print(" ", st)
    if len(missing_on_disk) > 50:
        print(f" ... ({len(missing_on_disk) - 50} more)")

if extra_on_disk:
    print("\n--- Found on disk but NOT in catalog (first 50) ---")
    for st in extra_on_disk[:50]:
        print(" ", st)
    if len(extra_on_disk) > 50:
        print(f" ... ({len(extra_on_disk) - 50} more)")

--------------- Disk vs catalog check ---------------
Stations in catalog: 604
Stations found on disk: 2070
Catalog stations missing on disk: 244
Stations on disk but not in catalog: 1710

--- In catalog but NOT found on disk (first 50) ---
  abby
  abot
  al2h
  alb4
  bbay
  bcab
  bcbu
  bcch
  bcco
  bccq
  bccr
  bccy
  bcdn
  bcdt
  bces
  bcho
  bclc
  bclg
  bcli
  bcmr
  bcnn
  bcns
  bcph
  bcpm
  bcsc
  bcsf
  bcsk
  bcsm
  bcsq
  bcts
  bcvc
  bcvi
  bcvt
  bcws
  bfir
  blnp
  blvu
  bndm
  brnb
  brsp
  bsum
  c046
  ca1r
  cach
  cacr
  cacy
  caeu
  cafl
  cafm
  cami
 ... (194 more)

--- Found on disk but NOT in catalog (first 50) ---
  108w
  1500
  7odm
  ab01
  ab02
  ab06
  ab07
  ab08
  ab09
  ab11
  ab12
  ab13
  ab14
  ab15
  ab17
  ab18
  ab21
  ab22
  ab25
  ab27
  ab28
  ab33
  ab35
  ab36
  ab37
  ab39
  ab41
  ab42
  ab43
  ab44
  ab45
  ab46
  ab48
  ab49
  ab50
  ab51
  ab53
  ac02
  ac03
  ac06
  ac07
  ac08
  ac09
  ac10
  ac11
  ac12
  ac13
  ac14
  ac

In [40]:
rows = {}
meta_rows = []
fail = []
multiple_matches = []
loaded = []

allowed_set = set(allowed_stations)
stations_to_load = sorted(allowed_set & stations_on_disk)  # <- intersection only

for st in stations_to_load:
    candidates = files_by_station.get(st, [])
    # candidates should exist by construction, but keep a guard anyway
    if not candidates:
        fail.append((st, "", "Station in stations_to_load but no files found (index bug)"))
        continue

    # Deterministic choice if multiple files exist
    if len(candidates) > 1:
        multiple_matches.append((st, [c.name for c in candidates]))
        candidates = sorted(
            candidates,
            key=lambda p: (0 if p.suffix.lower() == ".neu" else 1, p.name.lower())
        )

    chosen = candidates[0]

    try:
        df = parse_gps_file(chosen)
        rows[st] = df
        loaded.append(st)
    except Exception as e:
        fail.append((st, chosen.name, str(e)))

loaded_set = set(loaded)

print(f"Stations to load (catalog ∩ disk): {len(stations_to_load)}")
print(f"Loaded successfully: {len(loaded_set)}")
print(f"Failed: {len(fail)}")
print(f"Multiple matches: {len(multiple_matches)}")


Stations to load (catalog ∩ disk): 360
Loaded successfully: 360
Failed: 0
Multiple matches: 0


In [41]:
ds_list = []
for st, df in rows.items():
    ds_list.append(
        xr.Dataset({c: ("time", df[c].values) for c in df.columns},
                   coords={"time": df.index, "station": st}).expand_dims("station")
    )

ds = xr.concat(ds_list, dim="station").sortby("time")
ds

C:\Users\loicb\AppData\Local\Temp\ipykernel_35120\952775784.py:8: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  ds = xr.concat(ds_list, dim="station").sortby("time")


<xarray.Dataset> Size: 452MB
Dimensions:      (station: 360, time: 12062)
Coordinates:
  * station      (station) <U4 6kB 'agns' 'albh' 'aldr' ... 'ybhb' 'yelm' 'yonc'
  * time         (time) int64 96kB 0 1 2 3 4 5 ... 12057 12058 12059 12060 12061
Data variables: (12/13)
    Dec_Yr       (station, time) float64 35MB 2.021e+03 2.021e+03 ... nan nan
    Yr           (station, time) float64 35MB 2.021e+03 2.021e+03 ... nan nan
    DayOfYr      (station, time) float64 35MB 169.0 170.0 171.0 ... nan nan nan
    N            (station, time) float64 35MB 1.78 3.89 -0.41 ... nan nan nan
    E            (station, time) float64 35MB -3.05 -2.72 -2.69 ... nan nan nan
    U            (station, time) float64 35MB -5.53 0.77 -2.13 ... nan nan nan
    ...           ...
    E_sig        (station, time) float64 35MB 1.78 1.78 1.78 ... nan nan nan
    U_sig        (station, time) float64 35MB 4.97 4.49 4.73 ... nan nan nan
    CorrNE       (station, time) float64 35MB 0.055 0.0365 0.0438 ... nan nan
    CorrNU       (station, time) float64 35MB -0.2077 -0.1865 ... nan nan
    CorrEU       (station, time) float64 35MB -0.0767 -0.0581 ... nan nan
    Chi_Squared  (station, time) float64 35MB 1.39 1.95 0.39 ... nan nan nan